# Gemma 4 Voice Calling Agent

A conversational voice agent powered by Gemma 4.
**You speak** into the mic, the agent **listens, thinks, and speaks back**.

### Prerequisites
Make sure the Gemma 4 server is running:
```bash
cd ~/gemma-server && source venv/bin/activate && python server.py
```

## 0. Install Dependencies

In [1]:
!pip install -q langchain langchain-openai langchain-core openai SpeechRecognition gTTS pydub

'pip' is not recognized as an internal or external command,
operable program or batch file.


## 1. Connect to Gemma 4

In [ ]:
import speech_recognition as sr
from gtts import gTTS
import io
from pydub import AudioSegment
from pydub.playback import play
from IPython.display import Audio, display
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

recognizer = sr.Recognizer()
SYSTEM_PROMPT = "You are Gemma, a helpful AI voice assistant. Keep your responses concise and conversational."

llm = ChatOpenAI(
    model="gemma-4-e2b-it",
    base_url="http://localhost:8000/v1",
    api_key="not-needed",
    temperature=0.7,
    max_tokens=256,
)

history = [SystemMessage(content=SYSTEM_PROMPT)]
print("Voice agent initialized and connected to Gemma 4.")

c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\pydub\utils.py:170: RuntimeWarning: Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work
  warn("Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work", RuntimeWarning)


Voice agent initialized and connected to Gemma 4.


: 

## 2. Voice Calling Agent

Run this cell to start a voice conversation with Gemma 4.
- Click the **record** button to speak
- The agent will **listen**, **think**, and **respond with voice**
- Conversation history is maintained across turns

In [3]:
def agent_respond(user_text):
    """Send user text to Gemma and stream the response to the screen."""
    history.append(HumanMessage(content=user_text))
    
    full_response = ""
    print("Gemma: ", end="", flush=True)
    
    # Use llm.stream instead of llm.invoke
    for chunk in llm.stream(history):
        content = chunk.content
        full_response += content
        print(content, end="", flush=True)
    
    print() # New line at the end
    history.append(AIMessage(content=full_response))
    return full_response

### 2.5 Voice Functions
Define how the agent listens and speaks.

In [4]:
AUDIO_FILE = r"C:\\Users\\User\\voice.wav"
whisper_model = None

def listen():
    """Transcribe the local audio file using faster-whisper."""
    global whisper_model
    from faster_whisper import WhisperModel
    import os
    
    if not os.path.exists(AUDIO_FILE):
        return f"Error: {AUDIO_FILE} not found."
        
    if whisper_model is None:
        print("Loading faster-whisper model...")
        # Use 'base' for a good balance of speed and accuracy
        whisper_model = WhisperModel("base", device="cpu", compute_type="int8")
    
    print(f"(Transcribing {AUDIO_FILE}...)")
    segments, info = whisper_model.transcribe(AUDIO_FILE, beam_size=5)
    text = " ".join([segment.text for segment in segments]).strip()
    return text

def speak(text):
    """Convert text to speech and display an audio widget with autoplay."""
    from gtts import gTTS
    from IPython.display import Audio, display
    import io
    
    tts = gTTS(text=text, lang='en')
    fp = io.BytesIO()
    tts.write_to_fp(fp)
    fp.seek(0)
    
    # Use IPython display for a browser-based player (avoids ffmpeg dependency)
    display(Audio(data=fp.read(), autoplay=True))

print("Voice functions (listen/speak) initialized with faster-whisper.")

Voice functions (listen/speak) initialized with faster-whisper.


## 3. Start a Call
Run this cell each time you want to speak. The agent will listen and respond.

In [12]:
try:
    # Listen
    user_text = listen()
    print(f"You: {user_text}")
    
    # Think
    print("Gemma is thinking...")
    response = agent_respond(user_text)
    # Streaming already printed the response
    
    # Speak
    speak(response)
    
except sr.WaitTimeoutError:
    print("No speech detected. Run the cell again.")
except sr.UnknownValueError:
    print("Could not understand audio. Try again.")
except Exception as e:
    print(f"Error: {e}")


(Listening...)
No speech detected. Run the cell again.


## 4. Use an Audio File Instead
If you don't have a microphone, use a pre-recorded audio file.

In [ ]:
# Load audio file instead of microphone
AUDIO_FILE = r"C:\Users\User\voice.wav"

print("Your audio:")
from IPython.display import Audio, display
display(Audio(filename=AUDIO_FILE))

# Transcribe using faster-whisper
from faster_whisper import WhisperModel

print("Transcribing with faster-whisper...")
model = WhisperModel("base", device="cpu", compute_type="int8")
segments, info = model.transcribe(AUDIO_FILE, beam_size=5)

user_text = " ".join([segment.text for segment in segments]).strip()

print(f"\nYou said: {user_text}")

# Get Gemma's response
print("\nGemma is thinking...")
response = agent_respond(user_text)

print("\nOriginal response:")
print(response)

# ---------------- PREPROCESSING STEP ----------------
# Remove punctuation before speaking
import re

clean_response = re.sub(r"[^\w\s]", "", response)

print("\nCleaned response for TTS:")
print(clean_response)

# Speak it back
speak(clean_response)

Your audio:


Transcribing with faster-whisper...

You said: We are hiring. Please see our openings at MySignatureCare.org slash careers.  We are also hosting many hiring fairs around the area.  Come talk to a recruiter in person or meet with a team member virtually.  Visit MySignatureCare.org slash careers or follow us on Instagram at SignatureHLTH to learn more.  Know where to get care when you need it.  In the event of an emergency, such as a heart attack or severe head injury, call 911.  For non-emergency situations, please call your primary care provider.  There are times when an urgent care may be appropriate for your health care needs.  Some of these situations might include a sore throat, cut sore lacerations requiring minor stitches,  ear and eye infections.  Our urgent care center wants to connect your needs back to your primary care provider  by sharing information about your visit with your doctor.

Gemma is thinking...
Gemma: Hey there! 👋 It sounds like MySignatureCare is hiring and als

: 

## 5. Continuous Conversation Loop
Run this for a multi-turn phone call experience. Say **"goodbye"** to hang up.

In [6]:
import time

print("Starting call... Say 'goodbye' or 'bye' to hang up.")
speak("Hello! I'm Gemma, your AI assistant. How can I help you today?")

for _ in range(1): # Limited to 1 turn for demo/testing
    try:
        time.sleep(1)  # Brief pause between turns
        user_text = listen()
        print(f"You: {user_text}")
        
        # Check for hang up
        if any(word in user_text.lower() for word in ['goodbye', 'bye', 'hang up', 'end call']):
            print("Gemma: Goodbye! It was nice talking to you.")
            speak("Goodbye! It was nice talking to you.")
            break
        
        response = agent_respond(user_text)
        # Streaming already printed the response
        speak(response)
        
    except sr.WaitTimeoutError:
        print("(silence detected, still listening...)")
        continue
    except sr.UnknownValueError:
        print("(didn't catch that, still listening...)")
        continue
    except KeyboardInterrupt:
        print("\nCall ended.")
        break

print(f"\nCall summary: {len(history) - 1} messages exchanged.")

Starting call... Say 'goodbye' or 'bye' to hang up.


Loading faster-whisper model...
(Transcribing C:\Users\User\voice.wav...)
You: We are hiring. Please see our openings at MySignatureCare.org slash careers.  We are also hosting many hiring fairs around the area.  Come talk to a recruiter in person or meet with a team member virtually.  Visit MySignatureCare.org slash careers or follow us on Instagram at SignatureHLTH to learn more.  Know where to get care when you need it.  In the event of an emergency, such as a heart attack or severe head injury, call 911.  For non-emergency situations, please call your primary care provider.  There are times when an urgent care may be appropriate for your health care needs.  Some of these situations might include a sore throat, cut sore lacerations requiring minor stitches,  ear and eye infections.  Our urgent care center wants to connect your needs back to your primary care provider  by sharing information about your visit with your doctor.
Gemma: Hey there! 👋 It sounds like MySignatureCare is hiri


Call summary: 4 messages exchanged.


## 6. View Conversation History
See the full transcript of your call.

In [25]:
print("=" * 50)
print("CALL TRANSCRIPT")
print("=" * 50)
for msg in history[1:]:  # skip system prompt
    role = "You" if msg.type == "human" else "Gemma"
    print(f"{role}: {msg.content}")
    print("-" * 30)

CALL TRANSCRIPT


## 7. Reset Conversation
Clear history to start a fresh call.

In [24]:
history = [SystemMessage(content=SYSTEM_PROMPT)]
print("Conversation history cleared. Ready for a new call.")

Conversation history cleared. Ready for a new call.
